<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/05-loss-optimization-training-dynamics.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **Loss Functions, Optimization, and Training Dynamics** {#loss-optimization-training-dynamics}

Chapter 04 explained how reverse-mode automatic differentiation turns a scalar objective into gradients. Training still has two decisions left: **what scalar objective should represent useful behavior**, and **how should its gradients change the parameters over many noisy steps**? A loss specifies the learning signal; an optimizer specifies the update dynamics. They are coupled: a perfectly implemented optimizer can optimize an unsuitable objective, while a well-designed loss can fail under an unstable learning rate, poor initialization, or a mismatched batch regime.

This chapter follows one training step from objective design to diagnostics. It distinguishes data fit from regularization, derives the update rules behind SGD, momentum, adaptive methods, and AdamW, and then connects those equations to choices that matter in practical PyTorch training: schedules, warmup, accumulation, clipping, mixed precision, initialization, and measurement.

### **Objectives, Empirical Risk, and Learning Signals** {#objectives-empirical-risk-learning-signals}

For a dataset $\mathcal{D}=\{(x_i,y_i)\}_{i=1}^{N}$, supervised training commonly minimizes a regularized empirical objective

$$
J(\theta)
=
\underbrace{\frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(x_i),y_i)}_{\text{data fit / empirical risk}}
+
\underbrace{\lambda\,\Omega(\theta)}_{\text{regularization or prior preference}}.
$$

The loss $\ell$ converts one prediction-target pair into a scalar discrepancy. The dataset average estimates expected risk under the observed data distribution. The regularizer expresses a preference that is not supplied by a single example, such as smaller weights, smoother functions, sparse features, or agreement across augmented views. In a probabilistic model, minimizing negative log-likelihood corresponds to maximizing the probability assigned to observed data; in discriminative training, the same scalar can be viewed more operationally as the signal whose gradient changes parameters.

The objective is a proxy, not the final product goal. Cross-entropy rewards calibrated probability assigned to a labeled class, but it does not directly encode fairness, latency, abstention, ranking utility, or the cost of a particular error. Before changing the optimizer, verify that the scalar objective actually represents the behavior that matters. A steadily decreasing training loss only proves that this particular proxy is being optimized on the observed samples.

Reduction is part of objective design. With a `mean` reduction, duplicating every example leaves the parameter gradient unchanged; with a `sum` reduction, it multiplies the gradient by the duplication count. This is why learning rates, gradient accumulation, and distributed data-parallel averaging must be interpreted together with the loss reduction.

<details>
<summary><strong>PyTorch: inspect empirical-risk reduction and an L2 preference</strong></summary>

~~~python
import copy
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(51)
x = torch.randn(6, 3)
targets = torch.tensor([0, 1, 2, 1, 0, 2])
template = nn.Linear(3, 3, bias=False)


def parameter_gradient(reduction: str, repeats: int) -> torch.Tensor:
    model = copy.deepcopy(template)
    logits = model(x.repeat((repeats, 1)))
    repeated_targets = targets.repeat(repeats)
    data_fit = F.cross_entropy(logits, repeated_targets, reduction=reduction)
    l2_preference = 0.05 * model.weight.square().sum()
    objective = data_fit + l2_preference
    objective.backward()
    return model.weight.grad.detach()


mean_once = parameter_gradient("mean", repeats=1)
mean_twice = parameter_gradient("mean", repeats=2)
sum_once = parameter_gradient("sum", repeats=1)
sum_twice = parameter_gradient("sum", repeats=2)
l2_gradient = 0.1 * template.weight.detach()

# The regularizer is unchanged, so compare the data term separately below.
assert torch.allclose(mean_once - l2_gradient, mean_twice - l2_gradient)
assert torch.allclose(sum_twice - l2_gradient, 2.0 * (sum_once - l2_gradient))

print("mean-reduced data-gradient norm:", float((mean_once - l2_gradient).norm()))
print("sum-reduced data-gradient ratio:", float((sum_twice - l2_gradient).norm() / (sum_once - l2_gradient).norm()))
~~~

</details>

The L2 gradient in the example is $2\lambda W$, so it is present even if the data gradient is zero. Optimizers should not silently decide which behavior is preferred: the loss defines the target, while the optimizer determines the path used to approach it.

**Application.** This framing applies to classifiers, regressors, language models, retrieval systems, diffusion models, and reinforcement-learning objectives. In each case, first write down the exact scalar being averaged, masked, weighted, or regularized.

**Comparison summary.** A per-example loss says what constitutes an error; empirical risk aggregates observed errors; regularization adds preferences beyond an individual example; the optimizer only acts on the resulting gradient.

### **Regression and Classification Losses** {#regression-classification-losses}

For regression target $y\in\mathbb{R}$ and prediction $\hat y$, common losses make different assumptions about noise and outliers:

$$
\ell_{\mathrm{MSE}}=(\hat y-y)^2,
\qquad
\ell_{\mathrm{MAE}}=|\hat y-y|,
$$

$$
\ell_{\mathrm{Huber},\delta}(r)=
\begin{cases}
\frac{1}{2}r^2, & |r|\leq\delta,\\
\delta\left(|r|-\frac{1}{2}\delta\right), & |r|>\delta,
\end{cases}
\qquad r=\hat y-y.
$$

MSE is the negative log-likelihood objective for Gaussian observation noise with fixed variance. Its gradient grows linearly with residual magnitude, so a large mistake receives disproportionately large influence. MAE is more robust to outliers but has a nondifferentiable point at zero and a constant-magnitude gradient away from it. Huber loss is quadratic near a good fit and linear for large residuals, making it a useful compromise when labels are mostly reliable but occasional large errors exist.

For classification, train on **logits** rather than already normalized probabilities. In binary classification, a logit $z$ becomes $p=\sigma(z)$ and binary cross-entropy is

$$
\ell_{\mathrm{BCE}}(z,y)
=-\left[y\log\sigma(z)+(1-y)\log(1-\sigma(z))\right].
$$

For $K$ classes, softmax cross-entropy is

$$
p_k=\frac{e^{z_k}}{\sum_j e^{z_j}},
\qquad
\ell_{\mathrm{CE}}(z,y)=-\log p_y.
$$

The practical API `binary_cross_entropy_with_logits` or `cross_entropy` fuses normalization with the log-loss using stable `logsumexp` algebra. Applying `sigmoid` or `softmax` first and then taking logs can underflow or overflow for confident mistakes. The useful derivative remains simple: softmax cross-entropy sends $p-q$ back to logits for one-hot target $q$.

<details>
<summary><strong>PyTorch: compare outlier gradients and stable logit losses</strong></summary>

~~~python
import torch
from torch.nn import functional as F

predictions = torch.tensor([0.2, -0.5, 5.0], requires_grad=True)
targets = torch.zeros(3)

mse = F.mse_loss(predictions, targets, reduction="none")
mae = F.l1_loss(predictions, targets, reduction="none")
huber = F.huber_loss(predictions, targets, delta=1.0, reduction="none")

mse.mean().backward(retain_graph=True)
mse_gradient = predictions.grad.detach().clone()
predictions.grad.zero_()
huber.mean().backward()
huber_gradient = predictions.grad.detach().clone()

# The outlier's MSE gradient grows with its residual; Huber's is capped here.
assert mse_gradient[-1].abs() > huber_gradient[-1].abs()
assert torch.allclose(huber_gradient[-1], torch.tensor(1.0 / 3.0))

extreme_logits = torch.tensor([-100.0, 100.0])
binary_targets = torch.tensor([1.0, 0.0])
stable_bce = F.binary_cross_entropy_with_logits(extreme_logits, binary_targets)
probabilities = extreme_logits.sigmoid()
naive_bce = -(binary_targets * probabilities.log() + (1 - binary_targets) * (1 - probabilities).log()).mean()

assert torch.isfinite(stable_bce)
assert not torch.isfinite(naive_bce)
print("per-example MSE:  ", mse.detach().tolist())
print("per-example Huber:", huber.detach().tolist())
print("stable BCE:", float(stable_bce))
~~~

</details>

**Application.** Choose MSE when large deviations should be strongly penalized and Gaussian noise is plausible; Huber for robust numeric prediction; BCE-with-logits for independent binary labels; and multiclass cross-entropy for one mutually exclusive label per example. Multi-label classification uses one binary loss per label, not softmax cross-entropy across labels.

**Comparison summary.** Losses are modeling choices. MSE emphasizes large residuals, MAE limits their influence, Huber interpolates between them, and logit-based cross-entropy converts probabilistic classification errors into stable, informative gradients.

### **Sequence, Metric, and Imbalanced-Data Losses** {#sequence-metric-imbalanced-losses}

Many deep-learning tasks do not map one input to one independent label. A language model or sequence tagger produces logits $Z\in\mathbb{R}^{B\times T\times V}$: $B$ sequences, $T$ positions, and vocabulary or label size $V$. Token negative log-likelihood must ignore padding positions and often normalize by the number of valid tokens rather than by $B\times T$:

$$
\mathcal{L}_{\mathrm{token}}
=-\frac{1}{\sum_{b,t}m_{bt}}
\sum_{b,t}m_{bt}\log p_\theta(y_{bt}\mid x_b,y_{b,<t}),
$$

where $m_{bt}\in\{0,1\}$ is the attention or loss mask. Without masking, extra padding changes the objective and rewards a model for predicting a padding token rather than useful content.

Label smoothing replaces one-hot target $q$ with a slightly softened distribution, reducing pressure to make every training prediction infinitely confident. Focal loss downweights easy examples, commonly using $(1-p_t)^\gamma\ell_{\mathrm{CE}}$, while class-weighted cross-entropy increases the cost of errors on underrepresented classes. These tools address different symptoms: smoothing is chiefly about confidence and regularization; focal loss changes which examples dominate the gradient; class weights encode asymmetric class importance.

Reweighting also changes the population objective. A classifier trained with inverse-frequency weights is no longer estimating probabilities under the original class distribution unless an appropriate correction is made. Its ranking may improve for a rare class while the raw probabilities become poorly calibrated. Consequently, evaluate the intended operating threshold, per-class precision/recall, and calibration on an unweighted validation distribution rather than assuming that a lower weighted loss means better deployment decisions.

Metric-learning objectives define a relation between examples rather than only a class label. For a positive pair $(a,p)$ and negative $n$, a triplet-style objective asks for the positive distance to be smaller by a margin:

$$
\ell_{\mathrm{triplet}}
=\max\bigl(0, d(h_a,h_p)-d(h_a,h_n)+m\bigr).
$$

Contrastive objectives such as InfoNCE instead make a matched representation more similar than other candidates in a batch. Their effectiveness depends strongly on augmentation, negative selection, temperature, and batch construction, so the loss cannot be selected independently of the data pipeline.

<details>
<summary><strong>PyTorch: compute masked token loss, label smoothing, and focal weighting</strong></summary>

~~~python
import torch
from torch.nn import functional as F

torch.manual_seed(53)
B, T, V = 2, 4, 5
logits = torch.randn(B, T, V)
targets = torch.tensor([[1, 3, 0, -100], [2, 4, -100, -100]])
valid_tokens = targets.ne(-100)

# PyTorch flattens token positions and ignores padding through ignore_index.
masked_ce = F.cross_entropy(
    logits.reshape(-1, V),
    targets.reshape(-1),
    ignore_index=-100,
    label_smoothing=0.1,
)

safe_targets = targets.masked_fill(~valid_tokens, 0)
per_token_ce = F.cross_entropy(logits.transpose(1, 2), safe_targets, reduction="none")
probability_of_target = logits.softmax(dim=-1).gather(-1, safe_targets.unsqueeze(-1)).squeeze(-1)
focal_weight = (1.0 - probability_of_target).square()
masked_focal = (per_token_ce * focal_weight * valid_tokens).sum() / valid_tokens.sum()

assert torch.isfinite(masked_ce)
assert torch.isfinite(masked_focal)
assert int(valid_tokens.sum()) == 5
print("smoothed masked token loss:", float(masked_ce))
print("focal-style masked loss:   ", float(masked_focal))
~~~

</details>

**Application.** Use masks for padded language, speech, vision, and graph batches; use weighted or focal objectives only after confirming that the data distribution and evaluation metric justify asymmetric treatment; use metric losses for retrieval, verification, clustering, and representation learning.

**Comparison summary.** Sequence losses decide which positions count, class-imbalance losses decide which examples count more, and metric losses decide which relations between examples should be preserved. All three alter the gradient distribution, not merely the final scalar value.

### **Gradient Descent and Stochastic Gradients** {#gradient-descent-stochastic-gradients}

For a differentiable objective $J(\theta)$, gradient descent applies

$$
\theta_{t+1}=\theta_t-\eta_t\nabla J(\theta_t),
$$

where $\eta_t$ is the learning rate. Full-batch gradient descent computes the average over all $N$ examples before each update. It is conceptually clean but expensive for large datasets and can make only one update per full pass over data.

The learning rate must be interpreted relative to curvature. For the quadratic objective $J(\theta)=\frac{1}{2}\theta^\top H\theta$ with positive-definite Hessian $H$, fixed-step gradient descent is stable only when

$$
0<\eta<\frac{2}{\lambda_{\max}(H)}.
$$

Along an eigenvector with eigenvalue $\lambda_i$, the error is multiplied by $1-\eta\lambda_i$ each step. A rate above the bound makes at least one direction diverge; a very small rate is stable but slow. A large condition number $\kappa=\lambda_{\max}/\lambda_{\min}$ produces the familiar zigzag: the rate must respect the steep direction while progress along the shallow direction remains limited. Momentum and adaptive preconditioning are partly attempts to improve this geometry.

Stochastic gradient descent (SGD) uses one example or a small minibatch $\mathcal{B}_t$ instead:

$$
g_t=
\frac{1}{|\mathcal{B}_t|}\sum_{i\in\mathcal{B}_t}
\nabla_\theta\ell_i(\theta_t),
\qquad
\theta_{t+1}=\theta_t-\eta_tg_t.
$$

When batches are sampled appropriately, $g_t$ is an unbiased estimator of the full empirical gradient, but it has variance. That noise is not simply an implementation defect: it makes frequent inexpensive updates possible, can help move across shallow barriers, and interacts with generalization. It also means one noisy loss value or one update direction is weak evidence about whether training is healthy.

Minibatches are the practical compromise. They reduce gradient-estimator variance and exploit matrix hardware efficiently, while still producing more frequent updates than full-batch optimization. Increasing batch size changes both the statistical estimator and the systems behavior; it is not equivalent to a harmless speed setting.

<details>
<summary><strong>PyTorch: measure stochastic-gradient variance and train with minibatches</strong></summary>

~~~python
import torch

torch.manual_seed(57)
N = 256
features = torch.randn(N)
targets = 2.5 * features


def batch_gradient(weight: torch.Tensor, indices: torch.Tensor) -> torch.Tensor:
    prediction = weight * features[indices]
    return (2.0 * features[indices] * (prediction - targets[indices])).mean()


weight_at_start = torch.tensor(0.0)
generator = torch.Generator().manual_seed(58)
single_estimates = torch.stack([
    batch_gradient(weight_at_start, torch.randint(N, (1,), generator=generator))
    for _ in range(400)
])
minibatch_estimates = torch.stack([
    batch_gradient(weight_at_start, torch.randint(N, (32,), generator=generator))
    for _ in range(400)
])


def train_with_batch_size(batch_size: int) -> torch.Tensor:
    weight = torch.tensor(0.0)
    local_generator = torch.Generator().manual_seed(59 + batch_size)
    for _ in range(80):
        indices = torch.randint(N, (batch_size,), generator=local_generator)
        weight -= 0.1 * batch_gradient(weight, indices)
    return weight


single_solution = train_with_batch_size(1)
minibatch_solution = train_with_batch_size(32)
assert minibatch_estimates.var() < single_estimates.var()
assert abs(minibatch_solution - 2.5) < 0.1
print("single-example gradient variance:", float(single_estimates.var()))
print("minibatch gradient variance:     ", float(minibatch_estimates.var()))
print("learned weights (SGD, minibatch):", float(single_solution), float(minibatch_solution))
~~~

</details>

**Application.** Nearly all large neural networks use minibatch stochastic gradients. The same principle appears in online learning, streaming adaptation, and federated optimization, though their sampling assumptions and noise sources differ.

**Comparison summary.** Full-batch descent uses an exact dataset gradient but updates rarely; SGD uses a high-variance cheap estimate; minibatch SGD trades some randomness for more stable, hardware-efficient updates.

### **Momentum and Nesterov Acceleration** {#momentum-nesterov-acceleration}

Plain SGD reacts only to the current minibatch gradient. In a narrow curved valley, it can oscillate sharply across the steep direction while making slow progress along the shallow direction. **Momentum** keeps a velocity-like exponential moving average:

$$
v_t=\beta v_{t-1}+g_t,
\qquad
\theta_{t+1}=\theta_t-\eta v_t,
$$

where $\beta\in[0,1)$ controls memory. With this convention, recent consistent gradients build speed while rapidly changing, noisy directions partially cancel. Some libraries scale the new gradient by $(1-\beta)$; that changes the numerical scale of $v_t$ but not the central idea that the update remembers a decaying history.

**Nesterov accelerated gradient (NAG)** evaluates the gradient at an anticipated position instead of the current one. One common form is

$$
g_t=\nabla J(\theta_t-\eta\beta v_{t-1}),
\qquad
v_t=\beta v_{t-1}+g_t,
\qquad
\theta_{t+1}=\theta_t-\eta v_t.
$$

The lookahead asks, in effect, whether the current velocity is about to overshoot. It can begin correcting earlier than ordinary momentum, but it still requires an appropriate learning rate and does not eliminate all instability.

![Momentum aggregates gradients over time and reaches the minimum of an ill-conditioned quadratic more directly than an unaccelerated update.](assets/dl05-momentum-trajectory.svg){fig-align="center" width="72%" fig-alt="A contour plot of a two-dimensional quadratic optimization problem with a momentum trajectory approaching the minimum."}

*Image source: [Dive into Deep Learning, Momentum](https://d2l.ai/chapter_optimization/momentum.html), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*

<details>
<summary><strong>PyTorch: compare SGD, momentum, and Nesterov on an ill-conditioned quadratic</strong></summary>

~~~python
import torch

curvature = torch.tensor([1.0, 30.0])


def quadratic(theta: torch.Tensor) -> torch.Tensor:
    return 0.5 * (curvature * theta.square()).sum()


def optimize(method: str, steps: int = 80, lr: float = 0.02, beta: float = 0.9):
    theta = torch.tensor([4.0, 4.0])
    velocity = torch.zeros_like(theta)
    losses = []
    for _ in range(steps):
        if method == "nesterov":
            gradient = curvature * (theta - lr * beta * velocity)
        else:
            gradient = curvature * theta

        if method == "sgd":
            theta = theta - lr * gradient
        else:
            velocity = beta * velocity + gradient
            theta = theta - lr * velocity
        losses.append(quadratic(theta))
    return torch.stack(losses)


sgd_losses = optimize("sgd")
momentum_losses = optimize("momentum")
nesterov_losses = optimize("nesterov")

assert torch.isfinite(torch.stack((sgd_losses[-1], momentum_losses[-1], nesterov_losses[-1]))).all()
assert momentum_losses[-1] < sgd_losses[-1]
assert nesterov_losses[-1] < sgd_losses[-1]
print("final losses:", {"sgd": float(sgd_losses[-1]), "momentum": float(momentum_losses[-1]), "nesterov": float(nesterov_losses[-1])})
~~~

</details>

**Application.** Momentum SGD remains common in vision training and is often a strong baseline when carefully tuned. Nesterov momentum is useful when an implementation exposes it directly, but its gains depend on the curvature, batch noise, schedule, and other choices.

**Comparison summary.** SGD follows the current gradient only; momentum filters gradients through a velocity state; Nesterov momentum evaluates a lookahead gradient. Both accelerated methods add state and hyperparameters in exchange for faster movement in persistent directions.

### **Adaptive Optimizers** {#adaptive-optimizers}

An optimizer can do more than average gradients over time: it can also rescale coordinates. This is useful when different parameters receive gradients with dramatically different magnitudes. **AdaGrad** accumulates a nonnegative second-moment estimate for each coordinate,

$$
s_t=s_{t-1}+g_t\odot g_t,
\qquad
\theta_{t+1}=\theta_t-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}.
$$

Frequently active coordinates receive progressively smaller steps. This can be valuable for sparse features, but the denominator only grows, so learning can eventually become too slow.

**RMSProp** replaces the unbounded sum with an exponential moving average,

$$
s_t=\rho s_{t-1}+(1-\rho)g_t\odot g_t,
\qquad
\theta_{t+1}=\theta_t-\eta\frac{g_t}{\sqrt{s_t}+\epsilon}.
$$

The denominator acts as a diagonal preconditioner: large, repeatedly noisy coordinates are damped more than small ones. It is not a full curvature matrix, and it does not guarantee better generalization. Adaptive rescaling changes the optimization geometry, which is precisely why its learning-rate and weight-decay settings should not be copied blindly from SGD.

<details>
<summary><strong>PyTorch: inspect coordinate-wise scaling in AdaGrad and RMSProp</strong></summary>

~~~python
import torch

gradient = torch.tensor([100.0, 1.0])
learning_rate = 0.1
epsilon = 1e-8

sgd_update = learning_rate * gradient
adagrad_state = gradient.square()
adagrad_update = learning_rate * gradient / (adagrad_state.sqrt() + epsilon)

rho = 0.9
rmsprop_state = (1.0 - rho) * gradient.square()
rmsprop_update = learning_rate * gradient / (rmsprop_state.sqrt() + epsilon)

assert sgd_update[0] / sgd_update[1] == 100.0
assert torch.allclose(adagrad_update, torch.tensor([0.1, 0.1]), atol=1e-6)
assert torch.allclose(rmsprop_update[0], rmsprop_update[1], atol=1e-6)
print("SGD update:    ", sgd_update.tolist())
print("AdaGrad update:", adagrad_update.tolist())
print("RMSProp update:", rmsprop_update.tolist())
~~~

</details>

**Application.** Adaptive methods are often convenient defaults for transformers, language models, sparse problems, and fast prototyping. They are especially helpful when gradient scales vary across coordinates, but good results still require a schedule, sensible regularization, and validation against a task metric.

**Comparison summary.** Momentum averages first moments of gradients; AdaGrad accumulates all squared gradients; RMSProp exponentially averages squared gradients. These methods solve different conditioning and noise problems and can be combined, as Adam does.

### **AdamW and Decoupled Weight Decay** {#adamw-decoupled-weight-decay}

Adam combines first-moment momentum with an RMSProp-style second moment. Given gradient $g_t$, it forms

$$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,
\qquad
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2,
$$

then bias-corrects the initially zero estimates,

$$
\hat m_t=\frac{m_t}{1-\beta_1^t},
\qquad
\hat v_t=\frac{v_t}{1-\beta_2^t},
\qquad
\theta_{t+1}=\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.
$$

The usual default values, $\beta_1=0.9$, $\beta_2=0.999$, and $\epsilon=10^{-8}$, are starting points rather than laws. Because each coordinate has its own denominator, adding an L2 penalty gradient $\lambda\theta$ to Adam is not generally equivalent to multiplying parameters by a fixed decay factor. The adaptive denominator rescales the regularization signal too.

**AdamW** decouples these two operations:

$$
\theta'_{t}= (1-\eta\lambda)\theta_t,
\qquad
\theta_{t+1}=\theta'_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.
$$

The order varies slightly by implementation, but the key property is that weight decay acts directly on the parameter value rather than entering Adam's adaptive moment estimates. Exclude parameters such as bias vectors and normalization scale/shift from decay unless there is a deliberate reason to include them.

Optimizer state is also a memory decision. For $P$ trainable scalar parameters stored in FP32, the following rough accounting excludes activations, temporary buffers, allocator overhead, and distributed copies:

| Optimizer | Persistent state per parameter | Approximate parameters + gradients + optimizer state |
|---|---:|---:|
| SGD | none | $8P$ bytes |
| Momentum SGD | one velocity | $12P$ bytes |
| AdaGrad / RMSProp | one squared-gradient accumulator | $12P$ bytes |
| Adam / AdamW | first and second moments | $16P$ bytes |

RMSProp with momentum needs another velocity tensor. Some mixed-precision implementations also retain an FP32 master copy of lower-precision parameters, adding roughly $4P$ bytes. This is why changing SGD to AdamW can materially reduce the model size or batch size that fits even when the forward architecture is unchanged.

<details>
<summary><strong>PyTorch: observe coupled L2 regularization versus AdamW decay with zero data gradient</strong></summary>

~~~python
import torch

learning_rate = 0.1
weight_decay = 0.1
adam_parameter = torch.nn.Parameter(torch.tensor([1.0]))
adamw_parameter = torch.nn.Parameter(torch.tensor([1.0]))

adam = torch.optim.Adam([adam_parameter], lr=learning_rate, weight_decay=weight_decay)
adamw = torch.optim.AdamW([adamw_parameter], lr=learning_rate, weight_decay=weight_decay)

# There is no data gradient. Any movement comes from regularization/decay.
adam_parameter.grad = torch.zeros_like(adam_parameter)
adamw_parameter.grad = torch.zeros_like(adamw_parameter)
adam.step()
adamw.step()

assert torch.allclose(adamw_parameter.detach(), torch.tensor([0.99]), atol=1e-6)
assert adam_parameter.detach() < adamw_parameter.detach()
print("Adam with coupled L2: ", float(adam_parameter.detach()))
print("AdamW decay:          ", float(adamw_parameter.detach()))
~~~

</details>

**Application.** AdamW is the usual optimizer baseline for transformer-style models and many modern pretrained networks. Its practical configuration is a bundle: learning rate, weight decay, parameter groups, warmup, total updates, gradient clipping, precision, and effective batch size all interact.

**Comparison summary.** Adam adapts both direction and scale using two moment estimates. AdamW preserves that adaptive update while applying a separate, predictable shrinkage to selected parameter groups.

### **Learning Rate Schedules and Warmup** {#learning-rate-schedules-warmup}

The learning rate controls the scale at which gradients become parameter changes. A fixed learning rate that is useful early in training can be too large near a solution, where it causes persistent oscillation; a rate small enough for late refinement can make early progress unnecessarily slow. A **learning-rate schedule** therefore makes $\eta_t$ a deliberate function of update count, epoch, validation behavior, or training phase.

Common schedules include step decay, linear decay, cosine decay, and validation-triggered reductions. Cosine decay smoothly lowers the rate from $\eta_{max}$ to $\eta_{min}$ over $T$ updates:

$$
\eta_t=\eta_{min}+\frac{1}{2}(\eta_{max}-\eta_{min})
\left(1+\cos\left(\pi\frac{t}{T}\right)\right).
$$

**Warmup** starts from a smaller rate and increases it over the first $W$ updates, often linearly. It is especially useful when random initialization, large effective batches, adaptive optimizer states, normalization statistics, or pretrained-model adaptation make the first large updates unreliable. A simple warmup followed by cosine decay is

$$
\eta_t=
\begin{cases}
\eta_{max}\frac{t+1}{W}, & t<W,\\
\eta_{min}+\frac{1}{2}(\eta_{max}-\eta_{min})
\left[1+\cos\left(\pi\frac{t-W+1}{T-W}\right)\right], & t\geq W.
\end{cases}
$$

The unit matters. A schedule defined in *optimizer updates* changes if gradient accumulation, data-parallel world size, dataset size, or batch size changes. Record total updates and warmup updates explicitly rather than describing a schedule only in epochs.

![Warmup raises the learning rate gradually before a cosine schedule cools it for late-stage refinement.](assets/dl05-warmup-cosine-schedule.svg){fig-align="center" width="72%" fig-alt="A line plot showing a short linear learning-rate warmup followed by a gradual cosine-shaped decline."}

*Image source: [Dive into Deep Learning, Learning Rate Scheduling](https://d2l.ai/chapter_optimization/lr-scheduler.html), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*

<details>
<summary><strong>PyTorch: construct and check a warmup-plus-cosine update schedule</strong></summary>

~~~python
import math
import torch


def warmup_cosine_lr(step: int, total_updates: int, warmup_updates: int, base_lr: float, final_lr: float) -> float:
    if not 0 <= step < total_updates:
        raise ValueError("step must be inside the training schedule")
    if step < warmup_updates:
        return base_lr * (step + 1) / warmup_updates
    progress = (step - warmup_updates + 1) / (total_updates - warmup_updates)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return final_lr + (base_lr - final_lr) * cosine


total_updates, warmup_updates = 20, 4
base_lr, final_lr = 3e-3, 1e-4
learning_rates = [warmup_cosine_lr(step, total_updates, warmup_updates, base_lr, final_lr) for step in range(total_updates)]

# LambdaLR can use the same update-indexed function in a real optimizer.
parameter = torch.nn.Parameter(torch.tensor(1.0))
optimizer = torch.optim.SGD([parameter], lr=base_lr)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda step: warmup_cosine_lr(step, total_updates, warmup_updates, base_lr, final_lr) / base_lr,
)

# Exercise the scheduler at optimizer-update granularity, not merely construct it.
observed_rates = [optimizer.param_groups[0]["lr"]]
for _ in range(1, total_updates):
    optimizer.zero_grad(set_to_none=True)
    parameter.grad = torch.zeros_like(parameter)
    optimizer.step()
    scheduler.step()
    observed_rates.append(optimizer.param_groups[0]["lr"])

assert learning_rates[0] < learning_rates[warmup_updates - 1] == base_lr
assert learning_rates[-1] == final_lr
assert all(left >= right for left, right in zip(learning_rates[warmup_updates - 1 :], learning_rates[warmup_updates:]))
assert torch.allclose(torch.tensor(observed_rates), torch.tensor(learning_rates), atol=1e-9)
print("first six learning rates:", [round(rate, 6) for rate in learning_rates[:6]])
print("last learning rate:      ", learning_rates[-1])
~~~

</details>

**Application.** Warmup plus cosine decay is a strong explicit baseline for many transformer and vision training runs. Step decay can remain effective for mature recipes, while plateau scheduling is useful when validation feedback rather than a fixed update budget determines progress.

**Comparison summary.** The optimizer determines how a gradient is preconditioned; the schedule determines how far that transformed gradient moves parameters at each phase. Warmup protects early dynamics, while decay supports later refinement.

### **Batch Size, Gradient Accumulation, and Gradient Clipping** {#batch-size-accumulation-clipping-mixed-precision}

The **effective batch size** is the number of examples whose gradients contribute to one optimizer update. If each device uses microbatch $b$, gradients are accumulated for $a$ iterations, and data parallelism uses $w$ workers, then

$$
B_{\mathrm{effective}}=b\times a\times w.
$$

Gradient accumulation makes a large effective batch possible when one microbatch is all that fits in memory. For a mean-reduced loss, divide each microbatch loss by $a$ before `.backward()` so that the accumulated gradient matches the gradient of the concatenated batch. Step the optimizer and scheduler only after the effective batch is complete; calling `zero_grad()` between microbatches discards the partial sum.

Increasing the effective batch usually reduces gradient noise but also reduces the number of updates per epoch. The popular linear-scaling heuristic, $\eta'\approx k\eta$ when batch size grows by $k$, is a recipe-dependent starting point rather than a law: curvature, optimizer choice, normalization, data redundancy, and the number of training updates can all break it. Large-batch runs often pair retuned learning rates with warmup, but the comparison is meaningful only when update budget and data exposure are stated.

**Gradient clipping** limits an unusually large update. Global norm clipping replaces gradient vector $g$ by

$$
g\leftarrow g\min\left(1,\frac{c}{\lVert g\rVert_2+\epsilon}\right),
$$

where $c$ is `max_norm`. It is particularly useful in recurrent, generative, and unstable early-training regimes. Clipping is a safety mechanism, not a substitute for a valid loss, stable initialization, or appropriate learning rate. If it activates on nearly every step, diagnose the cause rather than simply lowering the threshold.

**Automatic mixed precision (AMP)** reduces memory and can improve accelerator throughput by using lower precision for selected operations while keeping sensitive work in safer precision. With FP16, small gradients may underflow to zero. `torch.amp.GradScaler` scales the loss before backward, then unscales before the optimizer step; it can skip an update whose gradients contain non-finite values. Crucially, unscale before inspecting or clipping gradients. BF16 has the same exponent range as FP32 and normally does not need gradient scaling, though it still changes precision.

For FP16 accumulation, the operational order is: `zero_grad` once per effective batch $\rightarrow$ autocast the forward pass and loss $\rightarrow$ scale the loss and call `backward` for each microbatch $\rightarrow$ unscale once $\rightarrow$ inspect or clip gradients $\rightarrow$ `scaler.step` $\rightarrow$ `scaler.update` $\rightarrow$ advance an update-based scheduler. Do not update the scale or unscale repeatedly while the same accumulated gradient is still being formed.

![Measured AMP speedup relative to FP32 varies substantially across neural-network workloads.](assets/dl05-pytorch-amp-speedup.png){fig-align="center" width="72%" fig-alt="A bar chart comparing mixed-precision speedup over FP32 for BERT, GNMT, NCF, ResNet, SSD, Tacotron, Transformer XL, and WaveGlow workloads."}

*Image source: [PyTorch, What Every User Should Know About Mixed Precision Training in PyTorch](https://pytorch.org/blog/what-every-user-should-know-about-mixed-precision-training-in-pytorch/). The benchmark illustrates that AMP is a measured systems optimization, not a fixed speedup guarantee.*

Distributed data parallelism contributes another averaged gradient term and therefore changes the effective batch size. It belongs to the scalable-systems treatment in Chapter 19; at this stage, treat world size as part of the update definition rather than a deployment-only detail.

<details>
<summary><strong>PyTorch: match gradient accumulation to a large batch, then use AMP-safe clipping</strong></summary>

~~~python
import copy
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(61)
features = torch.randn(8, 3)
targets = torch.randn(8, 2)
template = nn.Linear(3, 2)
large_batch_model = copy.deepcopy(template)
accumulated_model = copy.deepcopy(template)

large_optimizer = torch.optim.SGD(large_batch_model.parameters(), lr=0.05)
accumulated_optimizer = torch.optim.SGD(accumulated_model.parameters(), lr=0.05)

# One full batch and two microbatches should produce the same clipped update.
large_optimizer.zero_grad(set_to_none=True)
large_loss = F.mse_loss(large_batch_model(features), targets)
large_loss.backward()
torch.nn.utils.clip_grad_norm_(large_batch_model.parameters(), max_norm=1.0)
large_optimizer.step()

accumulated_optimizer.zero_grad(set_to_none=True)
for micro_features, micro_targets in zip(features.chunk(2), targets.chunk(2)):
    micro_loss = F.mse_loss(accumulated_model(micro_features), micro_targets)
    (micro_loss / 2).backward()  # Divide by accumulation steps for mean-reduced loss.
torch.nn.utils.clip_grad_norm_(accumulated_model.parameters(), max_norm=1.0)
accumulated_optimizer.step()

assert all(torch.allclose(a, b, atol=1e-6) for a, b in zip(large_batch_model.parameters(), accumulated_model.parameters()))

# The same ordering is safe with AMP. It runs as ordinary FP32 when CUDA is unavailable.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_model = nn.Linear(3, 2).to(device)
amp_optimizer = torch.optim.AdamW(amp_model.parameters(), lr=1e-3)
amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler(device.type, enabled=amp_enabled)
amp_optimizer.zero_grad(set_to_none=True)

with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
    amp_loss = F.mse_loss(amp_model(features.to(device)), targets.to(device))
scaler.scale(amp_loss).backward()
scaler.unscale_(amp_optimizer)  # Gradients must be unscaled before clipping.
unscaled_norm = torch.nn.utils.clip_grad_norm_(amp_model.parameters(), max_norm=1.0)
scaler.step(amp_optimizer)
scaler.update()

assert torch.isfinite(torch.stack([parameter.detach().abs().max().cpu() for parameter in amp_model.parameters()])).all()
print("accumulation matches full batch:", True)
print("pre-clipping AMP gradient norm:", float(unscaled_norm))
~~~

</details>

**Application.** Accumulation is common in memory-limited language, vision, and multimodal training. AMP is a standard performance technique on modern accelerators. Both require careful accounting: the loss reduction, scale factor, clipping point, scheduler step, and optimizer step must refer to the same effective update.

**Comparison summary.** Batch size controls the amount of data behind one estimate; accumulation emulates a larger batch over time; clipping bounds an abnormal gradient; AMP changes arithmetic precision and therefore requires correct scaling and unscaling order.

### **Initialization and Early Training Dynamics** {#initialization-early-training-dynamics}

Initialization determines the first activations, gradients, and symmetry properties of a model. If two hidden units start with identical parameters and receive the same input, they remain identical under gradient descent and fail to learn distinct features. Random initialization breaks this symmetry. Its variance must also be matched to architecture: values that repeatedly shrink or grow through layers can create near-zero or exploding activations and gradients before optimization has a chance to help.

For a layer with fan-in $n_{in}$ and fan-out $n_{out}$, Xavier/Glorot initialization aims to preserve variance for roughly linear or symmetric activations, often using variance near

$$
\operatorname{Var}(W)\approx\frac{2}{n_{in}+n_{out}}.
$$

For ReLU-like activations, He/Kaiming initialization compensates for roughly half the activations being removed:

$$
\operatorname{Var}(W)\approx\frac{2}{n_{in}}.
$$

These are variance-preservation heuristics, not universal guarantees. Residual paths, normalization, embedding scale, depth, optimizer, and precision all change the early dynamics. The useful practice is to combine a sensible default with measurement: inspect activation scales, gradient norms, update-to-parameter ratios, and non-finite values during the first few hundred updates.

<details>
<summary><strong>PyTorch: contrast symmetry-preserving zero initialization with Kaiming initialization</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F


class SmallReLUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(6, 12)
        self.output = nn.Linear(12, 1)

    def forward(self, x):
        hidden = F.relu(self.hidden(x))
        return self.output(hidden), hidden


def initialize(model: nn.Module, mode: str) -> None:
    if mode == "zero":
        nn.init.zeros_(model.hidden.weight)
        nn.init.zeros_(model.output.weight)
    elif mode == "kaiming":
        # Match Kaiming to the hidden ReLU, but use Xavier for the linear output.
        nn.init.kaiming_normal_(model.hidden.weight, nonlinearity="relu")
        nn.init.xavier_normal_(model.output.weight)
    else:
        raise ValueError(mode)
    nn.init.zeros_(model.hidden.bias)
    nn.init.zeros_(model.output.bias)


torch.manual_seed(67)
inputs = torch.randn(32, 6)
targets = torch.randn(32, 1)
zero_model, kaiming_model = SmallReLUNet(), SmallReLUNet()
initialize(zero_model, "zero")
initialize(kaiming_model, "kaiming")

zero_prediction, zero_hidden = zero_model(inputs)
zero_loss = F.mse_loss(zero_prediction, targets)
zero_loss.backward()
kaiming_prediction, kaiming_hidden = kaiming_model(inputs)
kaiming_loss = F.mse_loss(kaiming_prediction, targets)
kaiming_loss.backward()

assert zero_hidden.var() == 0
assert zero_model.hidden.weight.grad.abs().sum() == 0
assert kaiming_hidden.var() > 0
assert kaiming_model.hidden.weight.grad.abs().sum() > 0
print("zero-init hidden variance:   ", float(zero_hidden.detach().var()))
print("Kaiming hidden variance:     ", float(kaiming_hidden.detach().var()))
print("Kaiming first-layer grad norm:", float(kaiming_model.hidden.weight.grad.norm()))
~~~

</details>

**Application.** Start with framework defaults or architecture-recommended initialization, then inspect early dynamics before tuning a sophisticated optimizer. This is particularly important when adding a new residual branch, custom normalization, unusual activation, or a very deep recurrent/state-space component.

**Comparison summary.** Initialization supplies the starting distribution; optimization supplies iterative movement. Poor initialization can make gradients uninformative from the first step, whereas well-scaled random initialization breaks symmetry and gives optimization usable signals.

### **Loss Landscapes and Optimization Diagnostics** {#loss-landscapes-optimization-diagnostics}

The phrase **loss landscape** refers to the objective value as a function of all parameters. In a modern network this space has millions or billions of dimensions, so a two-dimensional contour plot is only a local projection. Even so, the picture gives useful intuition: narrow valleys create oscillation, flat or poorly conditioned directions create slow progress, saddle-like regions can have small gradients without being good solutions, and different parameterizations can represent the same function at different locations.

Do not diagnose a real run from loss alone. A useful training ledger records at least:

| Signal | What it can reveal | Caution |
|---|---|---|
| train loss and task metric | objective progress and task-facing behavior | low train loss can coexist with poor validation behavior |
| validation loss and metric | generalization and overfitting | compare at a stable evaluation protocol |
| learning rate and update count | whether the intended schedule is active | log actual optimizer values, not only configuration |
| global/per-layer gradient norms | explosion, vanishing, or disconnected paths | norms must be interpreted with scale and layer type |
| parameter norm and update-to-parameter ratio | whether updates are negligible or destructive | compare across time, not one step in isolation |
| activation distributions and non-finite counts | saturation, dead units, overflow, or invalid data | inspect representative layers rather than every tensor |

Several symptoms have different interventions. Loss that becomes `NaN` immediately suggests invalid data, a numerical operation, AMP overflow, or an excessive update; persistent near-zero gradients suggest saturation, masking, a disconnected graph, or poor scaling; training improvement with worsening validation suggests overfitting or a mismatch between objective and evaluation. Reducing the learning rate can be useful, but it is not a universal diagnosis.

The most reliable debugging procedure changes one meaningful variable at a time on a small deterministic run. First overfit a tiny subset, then confirm the objective, data pipeline, update accounting, precision mode, and metric. Only after that should a large hyperparameter search be trusted.

<details>
<summary><strong>PyTorch: log loss, gradient norm, and relative update size during a small training run</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(71)
features = torch.randn(128, 4)
true_weight = torch.tensor([1.2, -0.8, 0.5, 1.0])
labels = ((features @ true_weight) > 0).float().unsqueeze(1)

model = nn.Sequential(nn.Linear(4, 8), nn.Tanh(), nn.Linear(8, 1))
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-2, weight_decay=1e-3)
ledger = []


def parameter_vector(module: nn.Module) -> torch.Tensor:
    return torch.cat([parameter.detach().flatten() for parameter in module.parameters()])


for step in range(40):
    before = parameter_vector(model)
    optimizer.zero_grad(set_to_none=True)
    loss = F.binary_cross_entropy_with_logits(model(features), labels)
    loss.backward()
    gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()
    after = parameter_vector(model)
    update_ratio = (after - before).norm() / before.norm().clamp_min(1e-12)
    ledger.append(
        {
            "step": step,
            "loss": float(loss.detach()),
            "gradient_norm": float(gradient_norm),
            "update_ratio": float(update_ratio),
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
    )

assert ledger[-1]["loss"] < ledger[0]["loss"]
assert all(torch.isfinite(torch.tensor(row["gradient_norm"])) for row in ledger)
print("first step:", ledger[0])
print("last step: ", ledger[-1])
~~~

</details>

**Application.** These diagnostics belong in every serious training loop, from a laptop experiment to a distributed pretraining run. The exact logging system can change, but the habit of observing optimization as a time series is transferable.

**Comparison summary.** A landscape is useful intuition, while a diagnostic ledger is operational evidence. Loss values show whether the chosen scalar decreases; gradient, update, activation, precision, and validation signals explain why a run is stable, stalled, or misleading.

### **Chapter Comparison and Summary** {#chapter-comparison-summary}

Training is the repeated composition of an objective, a stochastic gradient estimate, an update rule, and a measurement process. There is no universally best optimizer or loss independent of data, architecture, batch regime, precision, and evaluation target.

| Component | Main question | Typical mechanism | Important failure mode |
|---|---|---|---|
| Objective | What behavior is rewarded? | empirical risk plus explicit preferences | optimizing a proxy that misses the task goal |
| Regression/classification loss | How are prediction errors valued? | MSE, Huber, BCE-with-logits, cross-entropy | unstable probability calculations or mismatched labels |
| Sequence/metric loss | Which positions, examples, or relations matter? | masks, weights, smoothing, contrastive terms | padding, imbalance, or negative-sampling mistakes |
| Minibatch SGD | How is a dataset gradient estimated? | random batch average | high variance or incorrect reduction scaling |
| Momentum/Nesterov | How are persistent directions accelerated? | velocity and lookahead | overshoot from incompatible learning rate/state |
| Adaptive methods | How are coordinates rescaled? | second-moment diagonal preconditioning | copying SGD hyperparameters without retuning |
| AdamW | How are adaptive updates and shrinkage separated? | Adam moments plus decoupled decay | decaying unsuitable parameters or using coupled L2 by accident |
| Schedule/warmup | How does step scale change over time? | update-indexed learning rate | scheduling in epochs while update count changes |
| Accumulation/AMP/clipping | How is one effective update executed safely? | scaled microbatch gradients, unscale, clip, step | stepping, clipping, or scheduling at the wrong granularity |
| Initialization/diagnostics | Is the early training signal usable? | variance-aware initialization and logged statistics | mistaking a symptom for a root cause |

The main conclusions are:

1. A loss encodes a modeling and product decision; it is not a neutral implementation detail.
2. The loss reduction, batch size, accumulation count, and data-parallel world size jointly define gradient scale.
3. Stable logit-based losses avoid numerical failure and provide direct gradients for classification.
4. Masks, class weights, focal factors, and metric terms redistribute gradient influence across positions, examples, and relationships.
5. Minibatch gradients are noisy estimates; their noise, update frequency, and hardware efficiency are part of the training design.
6. Momentum averages gradient directions, while adaptive optimizers rescale coordinates using gradient history.
7. AdamW decouples weight decay from adaptive gradient normalization and should use deliberate parameter groups.
8. Learning-rate schedules and warmup are update-counted control policies, not optional cosmetic settings.
9. Accumulation, clipping, AMP, and distributed averaging must be ordered and scaled around the same effective update.
10. Initialization determines whether useful symmetry-breaking activations and gradients exist before the optimizer can help.
11. Diagnostics should track time series of loss, validation, gradients, updates, activations, precision, and non-finite values.

Chapter 06 studies what happens after a model fits the training objective: generalization, regularization, experiment design, and the evidence needed to decide whether a training improvement is real rather than a lucky artifact of one split or seed.